# Get example blood+immune datasets from CELLxGENE

In [1]:
# !pip install -U -q cellxgene-census

In [2]:
import os
from os.path import join

import anndata
import pandas as pd
import tqdm

## Datasets to download

In [3]:
from dataclasses import dataclass
from typing import List


@dataclass
class CxGCollection:
    collection_id: str
    dataset_ids: List[str]
    celltype_cols: List[str]
    cell_type_author: str  # this is the most fine-grained annotation provided by the author
    sample_id: str  # this sequencing sample ID

In [4]:
cxg_collections = [
    CxGCollection(
        collection_id="7d7cabfd-1d1f-40af-96b7-26a0825a306d",
        dataset_ids=["01ad3cd7-3929-4654-84c0-6db05bd5fd59"],
        celltype_cols=["ct1", "ct2", "ct3"],
        cell_type_author="ct2",
        sample_id="batch"
    ),
    CxGCollection(
        collection_id="ed9185e3-5b82-40c7-9824-b2141590c7f0",
        dataset_ids=[
            "21d3e683-80a4-4d9b-bc89-ebb2df513dde",
            "30cd5311-6c09-46c9-94f1-71fe4b91813c"
        ],
        celltype_cols=["author_cell_type"],
        cell_type_author="author_cell_type",
        sample_id="donor_id"
    ),
    CxGCollection(
        collection_id="03f821b4-87be-4ff4-b65a-b5fc00061da7_PBMC",
        dataset_ids=["2a498ace-872a-4935-984b-1afa70fd9886"],
        celltype_cols=["annotation_broad", "annotation_detailed", "annotation_detailed_fullNames"],
        cell_type_author="annotation_detailed",
        sample_id="sample_id"
    ),
    CxGCollection(
        collection_id="03f821b4-87be-4ff4-b65a-b5fc00061da7_Airway",
        dataset_ids=["edc8d3fe-153c-4e3d-8be0-2108d30f8d70"],
        celltype_cols=["Cell_type_annotation_level1", "Cell_type_annotation_level2", "Cell_type_annotation_level3"],
        cell_type_author="Cell_type_annotation_level3",
        sample_id="sample_id"
    ),
    CxGCollection(
        collection_id="ddfad306-714d-4cc0-9985-d9072820c530",
        dataset_ids=["c7775e88-49bf-4ba2-a03b-93f00447c958"],
        celltype_cols=["author_cell_type"],
        cell_type_author="author_cell_type",
        sample_id="sample_id"
    ),
    CxGCollection(
        collection_id="b9fc3d70-5a72-4479-a046-c2cc1ab19efc",
        dataset_ids=["96a3f64b-0ee9-40d8-91e9-813ce38261c9"],
        celltype_cols=["Cell.group", "Cell.class"],
        cell_type_author="Cell.class",
        sample_id="sample"
    ),
    CxGCollection(
        collection_id="4f889ffc-d4bc-4748-905b-8eb9db47a2ed",
        dataset_ids=["de2c780c-1747-40bd-9ccf-9588ec186cee"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        sample_id="Sample ID"
    ),
    CxGCollection(
        collection_id="ced320a1-29f3-47c1-a735-513c7084d508",
        dataset_ids=["b0e547f0-462b-4f81-b31b-5b0a5d96f537"],
        celltype_cols=["author_cell_type"],
        cell_type_author="author_cell_type",
        sample_id="sample_uuid"
    ),
    # CxGCollection(
    #     collection_id="ced320a1-29f3-47c1-a735-513c7084d508_CAP",
    #     dataset_ids=["public-anndata_project-232_1___published__87833b35-f160-46eb-a827-b7593c473084.AIDAv2DataFreezeCT"],
    #     celltype_cols=["author_cell_type"],
    #     cell_type_author="author_cell_type",
    #     sample_id="sample_uuid"
    # ),
    CxGCollection(
        collection_id="b0cf0afa-ec40-4d65-b570-ed4ceacc6813",
        dataset_ids=["ed5d841d-6346-47d4-ab2f-7119ad7e3a35"],
        celltype_cols=["celltype.l1", "celltype.l2", "celltype.l3"],
        cell_type_author="celltype.l3",
        sample_id="orig.ident"
    ),
    CxGCollection(
        collection_id="eb735cc9-d0a7-48fa-b255-db726bf365af",
        dataset_ids=["c2a461b1-0c15-4047-9fcb-1f966fe55100"],
        celltype_cols=["Annotation"],
        cell_type_author="Annotation",
        sample_id="Sample_ID"
    )
]

## Download raw datasets

In [5]:
from cellxgene_census import download_source_h5ad

In [6]:
# DOWNLOAD_PATH = "/mnt/dssfs02/dataset-similarity/raw"
DOWNLOAD_PATH = "/vol/data/dataset-similarity/raw"

In [7]:
for collection in tqdm.tqdm(cxg_collections):
    for dataset in collection.dataset_ids:
        save_path = join(DOWNLOAD_PATH, f"{dataset}.h5ad")
        if not os.path.isfile(save_path):
            download_source_h5ad(dataset, to_path=save_path, census_version="2023-12-15", progress_bar=False)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 30593.03it/s]


## Preprocess datasets

In [8]:
import h5py
import numpy as np
from anndata.experimental import read_elem
from pandas.api.types import is_string_dtype
from scipy.sparse import csc_matrix, csr_matrix

In [9]:
# SAVE_PATH = "/mnt/dssfs02/dataset-similarity/preprocessed"
SAVE_PATH = "/vol/data/dataset-similarity/preprocessed"

In [10]:
def streamline_count_matrix(x_raw, gene_names_raw, gene_names_ref):
    assert len(gene_names_raw) == len(set(gene_names_raw))
    assert len(gene_names_ref) == len(set(gene_names_ref))
    assert len(gene_names_raw) == x_raw.shape[1]
    assert np.isin(gene_names_raw, gene_names_ref).sum() == x_raw.shape[1]
    # For fast column-wise slicing matrix has to be in csc format
    assert isinstance(x_raw, csc_matrix)
    gene_names_raw, gene_names_ref = np.array(gene_names_raw), np.array(gene_names_ref)
    row, col = np.empty(x_raw.nnz, dtype='i8'), np.empty(x_raw.nnz, dtype='i8')
    data = np.empty(x_raw.nnz, dtype='f4')

    ctr = 0    
    for i, gene in enumerate(gene_names_ref):
        if gene in gene_names_raw:
            gene_idx = np.where(gene == gene_names_raw)[0]
            assert gene_idx.size == 1
            gene_idx = gene_idx[0]
            x_col = x_raw[:, gene_idx]
            idxs_nnz = x_col.indices
            n_nnz = len(idxs_nnz)
            col[ctr:ctr+n_nnz] = i
            row[ctr:ctr+n_nnz] = idxs_nnz
            data[ctr:ctr+n_nnz] = x_col.data
            ctr += n_nnz

    return csr_matrix(
        (data, (row, col)),
        shape=(x_raw.shape[0], len(gene_names_ref)),
        dtype='f4'
    )


In [11]:
datasets = []
for collection in cxg_collections:
    for dataset in collection.dataset_ids:
        datasets.append(join(DOWNLOAD_PATH, f"{dataset}.h5ad"))

var_dfs = []
for dataset in datasets:
    with h5py.File(dataset) as f:
        var = read_elem(f["var"])[["feature_name"]]
        var.index.name = "feature_id"
        var_dfs.append(var)

var_concat = (
    pd.concat(var_dfs)
    .reset_index()
    .drop_duplicates()
    .set_index("feature_id")
    .sort_index()
)
var_concat = var_concat.reset_index().set_index("feature_name")
var_concat

,feature_id
feature_name,
TSPAN6,ENSG00000000003
TNMD,ENSG00000000005
DPM1,ENSG00000000419
SCYL3,ENSG00000000457
C1orf112,ENSG00000000460
...,...
RP11-484N11.1,ENSG00000288321
ATXN8,ENSG00000288330
RP4-733M16.8,ENSG00000288398


In [12]:
assert var_concat.index.is_unique
assert var_concat.feature_id.is_unique

var_concat.to_parquet(join(SAVE_PATH, "genes.parquet"))

In [13]:
# columns to keep for the preprocessed data
COLUMNS = [
    "assay", "cell_type", "development_stage", "disease", "donor_id", 
    "is_primary_data", "sex", "suspension_type", "tissue",
]


def preprocess_dataset(
    dataset_path: str, 
    var_set: pd.DataFrame, 
    columns: List[str], 
    cell_type_column: str,
    sample_id_column: str
):
    with h5py.File(dataset_path) as f:
        var = read_elem(f["var"])
        obs = read_elem(f["obs"])
        try:
            x = read_elem(f["raw"]["X"]).astype("f4").tocsc()
        except KeyError:
            # if raw doesn't exist -> use .X instead
            # according to CELLxGENE schema
            # https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/3.0.0/schema.md#x-matrix-layers
            x = read_elem(f["X"]).astype("f4").tocsc()

    # align feature spaces across datasets
    x = streamline_count_matrix(x, var.feature_name.tolist(), var_set.index.tolist())
    # subselect to desired obs columns
    obs = (
        obs[columns].copy()
        .assign(cell_type_author=lambda df: df[cell_type_column])
        .assign(sample_id=lambda df: df[sample_id_column])
    )
    # convert all columns with string dtype to categorical dtype
    for col in obs.columns:
        if is_string_dtype(obs[col]):
            obs[col] = obs[col].astype("category")

    return anndata.AnnData(X=x, obs=obs, var=var_set)


In [14]:
for collection in tqdm.tqdm(cxg_collections):
    save_path = join(SAVE_PATH, f"{collection.collection_id}.h5ad")
    if not os.path.isfile(save_path):
        adatas = []
        for dataset in collection.dataset_ids:
            columns = COLUMNS + collection.celltype_cols
            if collection.sample_id not in columns:
                columns.append(collection.sample_id)
            adata = preprocess_dataset(
                join(DOWNLOAD_PATH, f"{dataset}.h5ad"),
                var_concat,
                columns,
                collection.cell_type_author,
                collection.sample_id
            )
            adatas.append(adata)

        if len(adatas) > 1:
            adatas = anndata.concat(adatas)
            adatas.var = var_concat
        else:
            adatas = adatas[0]

        adatas.write(save_path, compression="gzip")


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 18517.90it/s]
